In [0]:
%pip install transformers
%pip install torch
%pip install torchvision

In [0]:
%restart_python

In [0]:
%sql
CREATE TABLE workspace.default.Product_reviews (
  review_id STRING,
  customer_name STRING,
  rating INT,
  review_text STRING,
  review_date DATE
)
USING DELTA;

In [0]:
%sql
INSERT INTO workspace.default.product_reviews VALUES
('001', 'John D.', 5,
'This product exceeded my expectations.

The build quality is top notch and feels very premium in hand. I was particularly impressed by the packaging - it was zero damage.

I have been using it for over a week now and there have been no issues whatsoever. Highly recommended.',
'2024-07-01'),
('002', 'Sara M.', 3,
'The product is decent for the price.

However, I was disappointed by the packaging. The box was partially torn and one corner of the item had scratches.

Overall, it works fine, but I expected better presentation.',
'2024-07-02'),
('003', 'Amir K.', 2,
'Very bad experience.

First of all, I received a [name] different product than what I ordered. The support team took 6 days to respond, and their reply was copy-paste templates.

Ultimately, I had to return the item. Waste of time and energy.',
'2024-03-01'),
('004', 'Priya S.', 4,
'Good product but with a few issues.

The quality is satisfactory and the features match the description. But delivery took longer than expected - almost 9 days.

Customer support was polite and kept me updated. Overall, a positive experience.',
'2024-07-24'),
('005', 'Alex M.', 2,
'Not very happy with the purchase.

Although the design looks sleek online, the actual product looks cheaper. Also, the instruction manual was missing.

It may be okay for casual use, but I wouldn’t buy it again.',
'2024-06-12')

In [0]:
%sql
Select * from workspace.default.product_reviews

In [0]:
%pip install --upgrade transformers

In [0]:
%restart_python

In [0]:
# Install if needed
# %pip install transformers torch mlflow

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-small"  # better than t5-small in Databricks

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def summarize(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    outputs = model.generate(
        inputs["input_ids"],
        max_length=50,
        min_length=10,
        num_beams=4
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test
text = "Databricks makes data teams productive with a unified analytics platform."
summary = summarize(text)

print(summary)

In [0]:
import mlflow
from mlflow.models.signature import infer_signature

# Example input/output
input_example = {"text": text}
output_example = {"summary": summary}

signature = infer_signature([text], [summary])

class SummarizerWrapper(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
        self.tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
        self.model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

    def predict(self, context, model_input):
        results = []
        for t in model_input["text"]:
            inputs = self.tokenizer(t, return_tensors="pt", truncation=True)
            outputs = self.model.generate(inputs["input_ids"], max_length=50)
            results.append(self.tokenizer.decode(outputs[0], skip_special_tokens=True))
        return results
mlflow.set_experiment("/Users/k.c.herur@gmail.com/kavita_rag_deploy_exp")
with mlflow.start_run():
    mlflow.pyfunc.log_model(
        name="kch_rag_deploy",
        python_model=SummarizerWrapper(),
        signature=signature,
        input_example={"text": [text]}
    )
#mlflow.get_experiment_by_name("/Users/kchung@databricks.com/rag").experiment_id

In [0]:
experiment_name = "/Users/k.c.herur@gmail.com/kavita_rag_deploy_exp"
model_artifact_path="Users/k.c.herur@gmail.com/project_databricks/simple-ai/rag_deployment/Models"
def get_or_create_experiment(experiment_name):
    experiment = mlflow.get_experiment_by_name(experiment_name)
    if experiment:
        return experiment.experiment_id
    else:
        return mlflow.create_experiment(experiment_name)

#experiment_name = "kavita_rag_deploy_exp"
experiment_id = get_or_create_experiment(experiment_name)
runs = mlflow.search_runs([experiment_id])

last_run_id = runs.sort_values("start_time", ascending=False).iloc[0].run_id
model_uri = f"runs:/{last_run_id}/{model_artifact_path}"
print(f"Model URI: {model_uri}")